In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType, ArrayType, MapType, DoubleType
from pyspark.sql import functions as F

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_Patient")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.master("local[*]")
.getOrCreate())

In [ ]:
bronze_path = "../../data_lake/bronze/hospital_data/resource_type=Organization/"
silver_base_path = "../../data_lake/silver/silver_organization/"

In [ ]:
general_coding_schema = StructType([
        StructField("system", StringType(), True),
        StructField("code", StringType(), True),
        StructField("display", StringType(), True)
    ])

type_schema = StructType([
        StructField("coding", ArrayType(general_coding_schema), True),
        StructField("text", StringType(), True)
    ])

identifier_schema = StructType([
        StructField("use", StringType(), True),
        StructField("system", StringType(), True),
        StructField("value",StringType(), True)
])

telecom_schema = StructType([
        StructField("system", StringType(), True),
        StructField("value", StringType(), True),
        StructField("use", StringType(), True)
])

extension_schema = StructType([
        StructField("url", StringType(), True),
        StructField("valueInteger", StringType(), True)
])

address_schema = StructType([
        StructField("line", ArrayType(StringType()), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("postalCode", StringType(), True),
        StructField("country", StringType(), True)
    ])

In [ ]:
def extract_extension_value(url_value):
    matches = F.filter(
        col("extension"),
        lambda x: x["url"].contains(url_value)
    )
    
    first_match = F.try_element_at(matches, F.lit(1))   # null if no match
    return first_match["valueInteger"]

def telecom_value(system_value):
    matches = F.filter(
        col("telecom"),
        lambda x: x["system"] == system_value
    )
    
    first_match = F.try_element_at(matches, F.lit(1))   # null if no match
    return first_match["value"]

In [ ]:
df_org = spark.read.format("parquet").load(bronze_path)

In [ ]:
df_inter1 = (
    df_org.select(
        col("fullUrl").alias("organization_fhir_id"),
        F.conv(F.substring(col("resource.id"),1, 15), 16, 10).cast("bigint").alias("organization_key"),
        col("resource.id").alias("organization_id"),
        from_json(col("resource.extension"), ArrayType(extension_schema)).alias("extension"),
        from_json(col("resource.identifier"), ArrayType(identifier_schema)).alias("identifier"),
        from_json(col("resource.type"), ArrayType(type_schema)).alias("type"),
        from_json(col("resource.telecom"), ArrayType(telecom_schema)).alias("telecom"),
        col("resource.name").alias("name"),
        col("resource.active").cast("boolean").alias("is_active"),
        from_json(col("resource.address"),ArrayType(address_schema)).alias("address")
        
    )
)

In [ ]:
df_inter2 = (
    df_inter1
    .withColumn("utilization_encounters", extract_extension_value("utilization-encounters-extension").cast("int"))
    .withColumn("utilization_procedures", extract_extension_value("utilization-procedures-extension").cast("int"))
    .withColumn("utilization_labs", extract_extension_value("utilization-labs-extension").cast("int"))
    .withColumn("utilization_prescriptions", extract_extension_value("utilization-prescriptions-extension").cast("int"))
    .withColumn("type_code",col("type")[0]["coding"][0]["code"])
    .withColumn("type_display", col("type")[0]["coding"][0]["display"] )
    .withColumn("telecom_phone", telecom_value("phone"))
    .withColumn("telecom_email", telecom_value("email"))
    .withColumn("telecom_fax",telecom_value("fax"))
    .withColumn("address_line",F.concat_ws(",",col("address")[0]["line"]))
    .withColumn("city",col("address")[0]["city"])
    .withColumn("state",col("address")[0]["state"])
    .withColumn("postalCode",col("address")[0]["postalCode"])
    .withColumn("country",col("address")[0]["country"])
    .withColumn("silver_timestamp", current_timestamp())
).drop("extension", "telecom","resource", "address","identifier","type")

In [ ]:
df_inter2.write.mode("overwrite").format("parquet").save(silver_base_path)